In [1]:
import torch
import numpy as np

In [2]:
class MyCube(torch.autograd.Function):
    @staticmethod
    def forward(x):
        # We wish to save dx for backward. In order to do so, it must
        # be returned as an output.
        dx = 3 * x ** 2
        result = x ** 3
        return result, dx

    @staticmethod
    def setup_context(ctx, inputs, output):
        x, = inputs
        result, dx = output
        ctx.save_for_backward(x, dx)

    @staticmethod
    def backward(ctx, grad_output, grad_dx):
        x, dx = ctx.saved_tensors
        # In order for the autograd.Function to work with higher-order
        # gradients, we must add the gradient contribution of `dx`,
        # which is grad_dx * 6 * x.
        result = grad_output * dx + grad_dx * 6 * x
        return result

# Wrap MyCube in a function so that it is clearer what the output is
def my_cube(x):
    result, dx = MyCube.apply(x)
    return result

In [ ]:
x = torch.randn(5, requires_grad=True)
y = my_cube(x)
torch.autograd.functional.jacobian(my_cube, x)

tensor([[ 1.7855,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000, 12.6098,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.3569,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0598,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  2.2499]])

In [23]:
x = torch.randn(5, requires_grad=True)
b = torch.randn(4, requires_grad=True)
A = torch.randn(4, 5, requires_grad=True)
func = lambda x, A, b: A @ x + b
torch.autograd.functional.jacobian(func, (x, A, b))[1] - torch.einsum('ij,k->ijk', torch.eye(4), x)

tensor([[[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0.]]], grad_fn=<SubBackward0>)

In [27]:
H = torch.rand_like(A)
torch.autograd.functional.jvp(lambda A: A @ x + b, inputs=(A, ), v=H)[1]

tensor([-0.9255, -2.1097, -2.0486, -1.4666])

In [32]:
H @ x

tensor([-0.9255, -2.1097, -2.0486, -1.4666], grad_fn=<MvBackward0>)

In [ ]:
# class AffineFunction(torch.autograd.Function):
#     @staticmethod
#     def forward(x, A, b):
#         return A @ x + b
    
#     @staticmethod
#     def setup_context(ctx, inputs, output):
#         x, A, b = inputs
        